In [0]:
## reading datasets
df_customers=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/first_volume/customers.csv")
df_sales=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/first_volume/sales.csv")

In [0]:

df_customers.printSchema()
df_sales.printSchema()
display(df_customers)
display(df_sales)

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: integer (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,Kevin,Larson,thomasmunoz@example.com,(736)633-3102x72821,1748 Caitlin Via Suite 145,South Katherineport,MD,80177
2,David,Griffin,jillalexander@example.org,692.707.7432x603,8294 Gregory Underpass Apt. 863,New Justintown,RI,40102
3,Jeffrey,Liu,lindsaywatts@example.org,380-970-4833x3990,91875 Fuller Field,Nicholeport,IN,55622
4,Amanda,Wilkinson,henryjessica@example.com,001-394-553-8642x24080,63572 Paul Meadow Apt. 088,Paulmouth,WV,96238
5,William,Cuevas,shawn94@example.com,(689)608-1555,874 Amber Tunnel Apt. 302,Wilsonville,MP,77543
6,Mark,Flores,schroedercody@example.com,(928)694-8612x753,220 Juan Bridge Suite 447,South Karenstad,ID,87307
7,Aaron,Mosley,richard29@example.com,435-409-1144,493 Nguyen Rue,South Gary,OR,34784
8,Jessica,Weaver,pamela65@example.com,001-264-405-8439x50136,3532 Taylor Knolls,North Hannah,VA,40883
9,Chad,Mccormick,hopkinsjimmy@example.org,+1-397-246-4369x857,4921 Rodriguez Islands Apt. 206,Tonyaland,MA,99043
10,William,Hernandez,justin00@example.net,881.235.4365x61025,708 Brittany Parks,West Jameshaven,NM,30628


sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,48,13,2026-05-20,3,1161.78
2,37,43,2026-04-09,1,205.02
4,25,41,2025-09-06,4,597.96
5,82,36,2025-09-19,3,1050.99
6,51,27,2025-10-28,5,236.45
7,7,39,2025-08-19,1,414.05
9,51,8,2025-08-03,2,983.82
10,12,32,2025-09-07,2,157.9
11,57,37,2025-11-18,4,1637.68
12,45,29,2026-04-14,2,629.86


## Queries in PySpark

In [0]:
#1. Total order amount for each customer
from pyspark.sql.functions import *

df=df_customers.join(df_sales, "customer_id") \
         .groupBy("customer_id","first_name","last_name") \
         .agg(sum("total_amount").alias("total_spend")) 
display(df)         

customer_id,first_name,last_name,total_spend
91,George,Hicks,3499.02
66,Joseph,Mcfarland,2756.45
15,Robert,Robinson,2658.22
52,Joshua,Gibbs,3262.83
59,Kevin,Williams,2037.26
9,Chad,Mccormick,2449.88
67,Mckenzie,Patton,666.6
48,Rachel,Ford,1765.84
68,William,Espinoza,4125.71
16,Rachel,Becker,1130.3400000000001


In [0]:
# 2. Top 3 customers by total spend

df.orderBy(df.total_spend.desc()).limit(3).show()

+-----------+-----------+---------+-----------------+
|customer_id| first_name|last_name|      total_spend|
+-----------+-----------+---------+-----------------+
|         54|    Melissa|    Smith|          7795.69|
|         87|      David|    Smith|6795.839999999999|
|         30|Christopher|  Vasquez|6584.049999999999|
+-----------+-----------+---------+-----------------+



In [0]:
# 3. Customers with no orders
df_customers.join(df_sales, "customer_id", "left") \
         .filter(col("sale_id").isNull()) \
         .show()

+-----------+----------+---------+--------------------+--------------------+--------------------+---------------+-----+--------+-------+----------+---------+--------+------------+
|customer_id|first_name|last_name|               email|        phone_number|             address|           city|state|zip_code|sale_id|product_id|sale_date|quantity|total_amount|
+-----------+----------+---------+--------------------+--------------------+--------------------+---------------+-----+--------+-------+----------+---------+--------+------------+
|          4|    Amanda|Wilkinson|henryjessica@exam...|001-394-553-8642x...|63572 Paul Meadow...|      Paulmouth|   WV|   96238|   NULL|      NULL|     NULL|    NULL|        NULL|
|          5|   William|   Cuevas| shawn94@example.com|       (689)608-1555|874 Amber Tunnel ...|    Wilsonville|   MP|   77543|   NULL|      NULL|     NULL|    NULL|        NULL|
|         13|   Vanessa|   Garcia|shelbybrock@examp...|          8899444109|429 Rodriguez Vil...|  P

In [0]:
# 4. City-wise total revenue
df_customers.join(df_sales, "customer_id") \
         .groupBy("city") \
         .agg(sum("total_amount").alias("city_revenue")) \
         .show()

+--------------------+------------------+
|                city|      city_revenue|
+--------------------+------------------+
|          North Alex|2320.8500000000004|
|        Timothymouth|           1347.77|
| South Katherineport|            568.94|
|          Edwardstad|           1351.11|
|       North Anthony|           1600.53|
|South Matthewborough|4037.2799999999997|
|        Heathermouth|           1765.84|
|           Lake Mary| 769.2399999999999|
|         Nicholeport|           3070.55|
|          Danielland|           1931.38|
|        West Rebecca|           1182.44|
|         Port Joseph|            515.61|
|            Chanfurt|            173.21|
|        Susanchester|           3114.07|
|         Brendaburgh|           2756.45|
|         South Megan|           2590.44|
|         New Richard|           2658.22|
|  Lake Christinefort|1742.6299999999999|
|         New Anthony|           3240.57|
|           Martinton|           4003.23|
+--------------------+------------

In [0]:
# 5. Average order amount per customer

df_customers.join(df_sales, "customer_id") \
         .groupBy("customer_id","first_name","last_name") \
         .agg(avg("total_amount").alias("average_order")) \
         .show()

+-----------+----------+---------+------------------+
|customer_id|first_name|last_name|     average_order|
+-----------+----------+---------+------------------+
|         91|    George|    Hicks|           699.804|
|         66|    Joseph|Mcfarland|          689.1125|
|         15|    Robert| Robinson| 886.0733333333333|
|         52|    Joshua|    Gibbs|          815.7075|
|         59|     Kevin| Williams| 679.0866666666667|
|          9|      Chad|Mccormick|           1224.94|
|         67|  Mckenzie|   Patton|             666.6|
|         48|    Rachel|     Ford|            882.92|
|         68|   William| Espinoza|           825.142|
|         16|    Rachel|   Becker|376.78000000000003|
|         41|   Melissa|    Parks|1069.5533333333333|
|         36|    Hannah|Carpenter|             97.63|
|         11|    Manuel| Andersen|          1271.955|
|         61|    Tricia| Robinson| 788.6300000000001|
|          8|   Jessica|   Weaver|           1793.45|
|         57|   Makayla|   C

In [0]:
# 6. Customers with more than one order
df_customers.join(df_sales, "customer_id") \
         .groupBy("customer_id","first_name","last_name") \
         .agg(count("sale_id").alias("order_count")) \
         .filter(col("order_count") > 1) \
         .show()

+-----------+----------+---------+-----------+
|customer_id|first_name|last_name|order_count|
+-----------+----------+---------+-----------+
|         91|    George|    Hicks|          5|
|         66|    Joseph|Mcfarland|          4|
|         15|    Robert| Robinson|          3|
|         52|    Joshua|    Gibbs|          4|
|         59|     Kevin| Williams|          3|
|          9|      Chad|Mccormick|          2|
|         48|    Rachel|     Ford|          2|
|         68|   William| Espinoza|          5|
|         16|    Rachel|   Becker|          3|
|         41|   Melissa|    Parks|          3|
|         11|    Manuel| Andersen|          2|
|         61|    Tricia| Robinson|          4|
|          8|   Jessica|   Weaver|          2|
|         57|   Makayla|   Cooper|          3|
|         38|   Jessica|   Fuller|          5|
|         23|    Brooke| Williams|          3|
|         60|    Bianca|   Powers|          3|
|         85|    Steven|   Norton|          2|
|         56|

In [0]:
# 7. Sort customers by total spend descending
df.orderBy(df.total_spend.desc()).show()

+-----------+-----------+---------+------------------+
|customer_id| first_name|last_name|       total_spend|
+-----------+-----------+---------+------------------+
|         54|    Melissa|    Smith|           7795.69|
|         87|      David|    Smith| 6795.839999999999|
|         30|Christopher|  Vasquez| 6584.049999999999|
|         74|      Sarah|   Flores|           6114.84|
|         33|      David|Mcfarland| 5770.639999999999|
|          7|      Aaron|   Mosley|           5621.52|
|         25|    Timothy|  Edwards| 5244.049999999999|
|         19|     Andrew|   Warren|           4966.17|
|         64|    Stephen|    Bruce|           4639.35|
|         63|     Isabel|   Martin|            4589.8|
|         55|       John|   Vargas|           4470.42|
|         38|    Jessica|   Fuller| 4456.139999999999|
|         68|    William| Espinoza|           4125.71|
|         23|     Brooke| Williams|4037.2799999999997|
|         57|    Makayla|   Cooper|           4003.23|
|         

## Queries in SQl

In [0]:
df_customers.createOrReplaceTempView("customers")
df_sales.createOrReplaceTempView("sales")

In [0]:
%sql
SELECT *
FROM customers;

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,Kevin,Larson,thomasmunoz@example.com,(736)633-3102x72821,1748 Caitlin Via Suite 145,South Katherineport,MD,80177
2,David,Griffin,jillalexander@example.org,692.707.7432x603,8294 Gregory Underpass Apt. 863,New Justintown,RI,40102
3,Jeffrey,Liu,lindsaywatts@example.org,380-970-4833x3990,91875 Fuller Field,Nicholeport,IN,55622
4,Amanda,Wilkinson,henryjessica@example.com,001-394-553-8642x24080,63572 Paul Meadow Apt. 088,Paulmouth,WV,96238
5,William,Cuevas,shawn94@example.com,(689)608-1555,874 Amber Tunnel Apt. 302,Wilsonville,MP,77543
6,Mark,Flores,schroedercody@example.com,(928)694-8612x753,220 Juan Bridge Suite 447,South Karenstad,ID,87307
7,Aaron,Mosley,richard29@example.com,435-409-1144,493 Nguyen Rue,South Gary,OR,34784
8,Jessica,Weaver,pamela65@example.com,001-264-405-8439x50136,3532 Taylor Knolls,North Hannah,VA,40883
9,Chad,Mccormick,hopkinsjimmy@example.org,+1-397-246-4369x857,4921 Rodriguez Islands Apt. 206,Tonyaland,MA,99043
10,William,Hernandez,justin00@example.net,881.235.4365x61025,708 Brittany Parks,West Jameshaven,NM,30628


In [0]:
%sql
select *from sales;

sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,48,13,2026-05-20,3,1161.78
2,37,43,2026-04-09,1,205.02
4,25,41,2025-09-06,4,597.96
5,82,36,2025-09-19,3,1050.99
6,51,27,2025-10-28,5,236.45
7,7,39,2025-08-19,1,414.05
9,51,8,2025-08-03,2,983.82
10,12,32,2025-09-07,2,157.9
11,57,37,2025-11-18,4,1637.68
12,45,29,2026-04-14,2,629.86


In [0]:
%sql
-- 1. total order amount for each customer
select
    c.customer_id,
    c.first_name,
    c.last_name,
    sum(s.total_amount) as total_spend
from customers c
join sales s
on c.customer_id = s.customer_id
group by
    c.customer_id,
    c.first_name,
    c.last_name;

customer_id,first_name,last_name,total_spend
91,George,Hicks,3499.02
66,Joseph,Mcfarland,2756.45
15,Robert,Robinson,2658.22
52,Joshua,Gibbs,3262.83
59,Kevin,Williams,2037.26
9,Chad,Mccormick,2449.88
67,Mckenzie,Patton,666.6
48,Rachel,Ford,1765.84
68,William,Espinoza,4125.71
16,Rachel,Becker,1130.3400000000001


In [0]:
%sql
-- 2. Top 3 customers by total spend
select
    c.customer_id,
    c.first_name,
    c.last_name,
    sum(s.total_amount) as total_spend
from customers c
join sales s
on c.customer_id = s.customer_id
group by
    c.customer_id,
    c.first_name,
    c.last_name
order by total_spend desc
limit 3;

customer_id,first_name,last_name,total_spend
54,Melissa,Smith,7795.69
87,David,Smith,6795.839999999999
30,Christopher,Vasquez,6584.049999999999


In [0]:
%sql
-- # 3. customers with no orders
select
    c.customer_id,
    c.first_name,
    c.last_name,
    s.sale_id
from customers c
left join sales s
on c.customer_id = s.customer_id
where s.sale_id is null;

customer_id,first_name,last_name,sale_id
4,Amanda,Wilkinson,null
5,William,Cuevas,null
13,Vanessa,Garcia,null
71,Stanley,Navarro,null
94,Mark,Rogers,null
95,John,Cherry,null
96,Jacob,Anderson,null
97,Stephanie,White,null
98,Charlotte,Baker,null
99,Samantha,Kelly,null


In [0]:
%sql
--  4. city-wise total revenue
select
    c.city,
    sum(s.total_amount) as city_revenue
from customers c
join sales s
on c.customer_id = s.customer_id
group by c.city;

city,city_revenue
North Alex,2320.8500000000004
Timothymouth,1347.77
South Katherineport,568.94
Edwardstad,1351.11
North Anthony,1600.53
South Matthewborough,4037.2799999999997
Heathermouth,1765.84
Lake Mary,769.2399999999999
Nicholeport,3070.55
Danielland,1931.38


In [0]:
%sql
-- 5. average order amount per customer
select
    c.customer_id,
    c.first_name,
    c.last_name,
    avg(s.total_amount) as average_order
from customers c
join sales s
on c.customer_id = s.customer_id
group by
    c.customer_id,
    c.first_name,
    c.last_name;

customer_id,first_name,last_name,average_order
91,George,Hicks,699.804
66,Joseph,Mcfarland,689.1125
15,Robert,Robinson,886.0733333333333
52,Joshua,Gibbs,815.7075
59,Kevin,Williams,679.0866666666667
9,Chad,Mccormick,1224.94
67,Mckenzie,Patton,666.6
48,Rachel,Ford,882.92
68,William,Espinoza,825.142
16,Rachel,Becker,376.78000000000003


In [0]:
%sql
-- 6. customers with more than one order
select
    c.customer_id,
    c.first_name,
    c.last_name,
    count(s.sale_id) as order_count
from customers c
join sales s
on c.customer_id = s.customer_id
group by
    c.customer_id,
    c.first_name,
    c.last_name
having count(s.sale_id) > 1;

customer_id,first_name,last_name,order_count
91,George,Hicks,5
66,Joseph,Mcfarland,4
15,Robert,Robinson,3
52,Joshua,Gibbs,4
59,Kevin,Williams,3
9,Chad,Mccormick,2
48,Rachel,Ford,2
68,William,Espinoza,5
16,Rachel,Becker,3
41,Melissa,Parks,3


In [0]:
%sql
-- 7. sort customers by total spend descending
select
    c.customer_id,
    c.first_name,
    c.last_name,
    sum(s.total_amount) as total_spend
from customers c
join sales s
on c.customer_id = s.customer_id
group by
    c.customer_id,
    c.first_name,
    c.last_name
order by total_spend desc;


customer_id,first_name,last_name,total_spend
54,Melissa,Smith,7795.69
87,David,Smith,6795.839999999999
30,Christopher,Vasquez,6584.049999999999
74,Sarah,Flores,6114.84
33,David,Mcfarland,5770.639999999999
7,Aaron,Mosley,5621.52
25,Timothy,Edwards,5244.049999999999
19,Andrew,Warren,4966.17
64,Stephen,Bruce,4639.35
63,Isabel,Martin,4589.8
